# BluaDiagnostics — Sprint 1 PoC
Demonstração Colab: system prompt + memória + function calling simulado.


In [ ]:
!pip install ollama python-dotenv -q


## Configuração
No Colab, configure `OLLAMA_API_KEY` em Secrets ou insira temporariamente via variável de ambiente.


In [ ]:
import os, json
# os.environ['OLLAMA_API_KEY'] = 'cole_sua_chave_aqui'  # não commitar chaves
os.environ.setdefault('OLLAMA_HOST', 'https://ollama.com')
os.environ.setdefault('MODEL_NAME', 'gpt-oss:120b')


In [ ]:
SYSTEM_PROMPT = '# System Prompt — BluaDiagnostics\n\n## PAPEL\nVocê é o BluaDiagnostics, um agente conversacional de apoio ao cuidado remoto da Care Plus/Blua. \nSeu papel é conduzir uma autoavaliação digital inicial, organizar informações clínicas relatadas pelo beneficiário e apoiar fluxos pós-teleconsulta com segurança.\n\n## PERSONA ATENDIDA\nBeneficiário final em autoavaliação digital.  \nA comunicação deve ser clara, empática, objetiva e sem alarmismo.\n\n## ESCOPO\nVocê pode:\n- coletar sintomas relatados pelo usuário;\n- perguntar sinais vitais informados manualmente ou simulados por wearable;\n- identificar sinais de alerta clínico;\n- consultar histórico simulado do paciente via ferramenta;\n- verificar interações medicamentosas simuladas via ferramenta;\n- sugerir encaminhamento para teleconsulta;\n- auxiliar na organização de informações para o médico.\n\n## RESTRIÇÕES\nVocê NÃO pode:\n- dar diagnóstico definitivo;\n- prescrever medicamentos sem aprovação médica;\n- substituir atendimento médico;\n- prometer cura ou resultado clínico;\n- ignorar sinais de alerta;\n- solicitar dados sensíveis desnecessários.\n\n## LGPD E PRIVACIDADE\nUse apenas dados necessários para a interação.\nNão exponha informações pessoais além do necessário.\nSempre explique que os dados são usados somente para apoio ao atendimento.\n\n## ESCALADA HUMANA\nEncaminhe imediatamente para atendimento humano ou emergência quando houver:\n- dor no peito;\n- falta de ar intensa;\n- desmaio;\n- confusão mental;\n- sinais neurológicos súbitos;\n- febre persistente com piora importante;\n- reação alérgica grave;\n- ideação suicida;\n- sintomas graves em gestantes, idosos ou crianças pequenas.\n\n## FORMATO DE SAÍDA\nResponda preferencialmente em formato estruturado:\n\nResumo:\n- ...\n\nSinais de alerta:\n- Sim/Não\n- Quais\n\nOrientação:\n- ...\n\nPróximo passo recomendado:\n- Auto cuidado / Teleconsulta / Urgência / Emergência\n\nObservação de segurança:\n- Esta orientação não substitui avaliação médica profissional.\n'


In [ ]:
PACIENTE = {
    'paciente_id': 'P001',
    'idade': 42,
    'condicoes': ['hipertensão leve', 'rinite alérgica'],
    'alergias': ['dipirona'],
    'medicamentos_em_uso': ['losartana 50mg']
}

def consultar_historico_paciente(paciente_id: str):
    return PACIENTE if paciente_id == 'P001' else {'erro': 'Paciente não encontrado'}

def verificar_interacoes_medicamentosas(medicamentos_novos, paciente_id: str):
    alertas = []
    for med in medicamentos_novos:
        m = med.lower()
        if 'dipirona' in m:
            alertas.append('Alergia registrada a dipirona.')
        if 'ibuprofeno' in m:
            alertas.append('Cautela: ibuprofeno pode exigir avaliação médica em paciente usando losartana.')
    return {'medicamentos': medicamentos_novos, 'alertas': alertas, 'status': 'avaliacao_medica_recomendada' if alertas else 'sem_alertas'}

def agendar_teleconsulta(paciente_id: str, especialidade: str, prioridade: str, motivo: str):
    return {'status': 'teleconsulta_simulada_agendada', 'paciente_id': paciente_id, 'especialidade': especialidade, 'prioridade': prioridade, 'motivo': motivo}


In [ ]:
def detectar_tool(mensagem):
    texto = mensagem.lower()
    if 'histórico' in texto or 'historico' in texto:
        return 'consultar_historico_paciente', {'paciente_id': 'P001'}
    if 'ibuprofeno' in texto or 'dipirona' in texto or 'medicamento' in texto:
        meds = []
        if 'ibuprofeno' in texto: meds.append('ibuprofeno')
        if 'dipirona' in texto: meds.append('dipirona')
        return 'verificar_interacoes_medicamentosas', {'medicamentos_novos': meds or ['medicamento informado'], 'paciente_id': 'P001'}
    if 'agendar' in texto or 'teleconsulta' in texto:
        return 'agendar_teleconsulta', {'paciente_id': 'P001', 'especialidade': 'clínica médica', 'prioridade': 'media', 'motivo': mensagem}
    return None, {}

def executar_tool(name, args):
    if name == 'consultar_historico_paciente': return consultar_historico_paciente(**args)
    if name == 'verificar_interacoes_medicamentosas': return verificar_interacoes_medicamentosas(**args)
    if name == 'agendar_teleconsulta': return agendar_teleconsulta(**args)
    return None


In [ ]:
from ollama import Client

client = Client(
    host=os.getenv('OLLAMA_HOST'),
    headers={'Authorization': 'Bearer ' + os.getenv('OLLAMA_API_KEY', '')}
)

memoria = []

def chat_blua(mensagem):
    tool_name, args = detectar_tool(mensagem)
    tool_result = executar_tool(tool_name, args) if tool_name else None
    contexto_tool = f'\nResultado da tool {tool_name}: {tool_result}' if tool_result else ''
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + memoria[-6:] + [{'role': 'user', 'content': mensagem + contexto_tool}]
    resp = client.chat(model=os.getenv('MODEL_NAME'), messages=messages, options={'temperature': 0.2, 'num_predict': 350}, stream=False)
    resposta = resp['message']['content']
    memoria.append({'role': 'user', 'content': mensagem})
    memoria.append({'role': 'assistant', 'content': resposta})
    return {'resposta': resposta, 'tool_chamada': tool_name, 'tool_result': tool_result}


## Demonstração com 3 turnos de memória e tools


In [ ]:
turnos = [
    'Quero fazer um check-up digital. Estou com dor de cabeça leve e pressão 12 por 8.',
    'Pode consultar meu histórico antes de eu falar com o médico?',
    'Uso losartana. Posso tomar ibuprofeno?'
]

for t in turnos:
    saida = chat_blua(t)
    print('\nUSUÁRIO:', t)
    print('TOOL:', saida['tool_chamada'])
    print('RESULTADO TOOL:', saida['tool_result'])
    print('ASSISTENTE:', saida['resposta'][:1200])
